<a href="https://colab.research.google.com/github/HemashreeVI/IBM_PDF_READER/blob/main/IBM_PDF_READER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install gradio openai faiss-cpu pypdf sentence-transformers numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 12.2 MB/s eta 0:00:00


In [ ]:
# =====================================================================
# PDF Q&A Bot v2 (Google Colab)  -  Gemini (default)/Groq/Together + FAISS + Gradio
# No LangChain needed, so it won't break on LangChain version changes.
#
# CELL 1 (run once):
#   !pip -q install gradio openai faiss-cpu pypdf sentence-transformers numpy
#
# API KEY: set PROVIDER below. When you run, a box asks for the key - paste it.
#          (Optional: store it as a Colab Secret instead. Never paste keys into code.)
#
# CELL 2: paste this whole file and run.
# =====================================================================
import os
import re
import getpass
import tempfile
from datetime import datetime

import numpy as np
import faiss
import gradio as gr
from pypdf import PdfReader
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------
# Pick ONE provider: "groq" (free tier), "gemini" (free tier) or "together" (needs deposit)
PROVIDER = "gemini"

PROVIDERS = {
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "secret": "GROQ_API_KEY",
        "models": ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"],
    },
    "gemini": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "secret": "GEMINI_API_KEY",
        "models": ["gemini-2.5-flash", "gemini-2.5-flash-lite"],
    },
    "together": {
        "base_url": "https://api.together.xyz/v1",
        "secret": "TOGETHER_API_KEY",
        "models": ["meta-llama/Llama-3.3-70B-Instruct-Turbo"],
    },
}
CFG = PROVIDERS[PROVIDER]
DEFAULT_MODEL = CFG["models"][0]   # editable in the UI
MODEL_CHOICES = CFG["models"]
EMBED_MODEL = "all-MiniLM-L6-v2"

SYSTEM_PROMPT = (
    "You are a careful assistant that answers questions about uploaded PDF documents. "
    "Use ONLY the provided context excerpts. Each excerpt is labelled like [file p.N]. "
    "Cite the labels you used, e.g. (report.pdf p.4). "
    "If the answer is not in the context, say you couldn't find it in the document "
    "instead of guessing. Be concise and clear."
)


def get_api_key() -> str:
    try:
        from google.colab import userdata
        key = userdata.get(CFG["secret"])
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get(CFG["secret"])
    return key or getpass.getpass(f"{PROVIDER} API key: ")


client = OpenAI(api_key=get_api_key(), base_url=CFG["base_url"])
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
print("Ready.")


# ---------------------------------------------------------------------
# Text chunking (sentence-aware, with overlap)
# ---------------------------------------------------------------------
def chunk_text(text: str, size: int = 1000, overlap: int = 200):
    text = re.sub(r"[ \t]+", " ", text).strip()
    if not text:
        return []
    pieces = [p.strip() for p in re.split(r"(?<=[.!?])\s+|\n{2,}", text) if p.strip()]

    # hard-split any piece that is longer than the chunk size
    flat = []
    for p in pieces:
        while len(p) > size:
            flat.append(p[:size])
            p = p[size:]
        flat.append(p)

    chunks, current = [], []
    cur_len = 0
    for p in flat:
        if cur_len + len(p) + 1 > size and current:
            chunks.append(" ".join(current))
            # carry over trailing sentences as overlap
            carry, carry_len = [], 0
            for s in reversed(current):
                if carry_len + len(s) > overlap:
                    break
                carry.insert(0, s)
                carry_len += len(s) + 1
            current, cur_len = carry, carry_len
        current.append(p)
        cur_len += len(p) + 1
    if current:
        chunks.append(" ".join(current))
    return chunks


# ---------------------------------------------------------------------
# Knowledge base
# ---------------------------------------------------------------------
class KnowledgeBase:
    def __init__(self):
        self.reset()

    def reset(self):
        self.chunks = []        # dicts: text, source, page
        self.embs = None        # (n, dim) float32
        self.index = None
        self.files = {}         # name -> {"pages": int, "chunks": int, "empty_pages": int}

    def add_pdf(self, path: str, chunk_size: int, overlap: int):
        name = os.path.basename(path)
        if name in self.files:
            return f"⚠️ {name} is already loaded."
        reader = PdfReader(path)
        if reader.is_encrypted:
            try:
                reader.decrypt("")
            except Exception:
                return f"❌ {name} is password-protected."

        new_chunks, empty_pages = [], 0
        for i, page in enumerate(reader.pages, start=1):
            try:
                txt = page.extract_text() or ""
            except Exception:
                txt = ""
            if not txt.strip():
                empty_pages += 1
                continue
            for c in chunk_text(txt, chunk_size, overlap):
                new_chunks.append({"text": c, "source": name, "page": i})

        if not new_chunks:
            return f"❌ {name}: no extractable text (it may be a scanned PDF that needs OCR)."

        embs = embedder.encode(
            [c["text"] for c in new_chunks],
            batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype("float32")

        self.chunks.extend(new_chunks)
        self.embs = embs if self.embs is None else np.vstack([self.embs, embs])
        self.index = faiss.IndexFlatIP(self.embs.shape[1])
        self.index.add(self.embs)
        self.files[name] = {
            "pages": len(reader.pages),
            "chunks": len(new_chunks),
            "empty_pages": empty_pages,
        }
        msg = f"✅ {name}: {len(reader.pages)} pages, {len(new_chunks)} chunks"
        if empty_pages:
            msg += f" ({empty_pages} pages had no text)"
        return msg

    def search(self, query: str, k: int = 4, only_file: str = None):
        if self.index is None:
            return []
        q = embedder.encode([query], normalize_embeddings=True).astype("float32")
        n = min(len(self.chunks), max(k * 5, k) if only_file else k)
        scores, ids = self.index.search(q, n)
        hits = []
        for s, i in zip(scores[0], ids[0]):
            if i < 0:
                continue
            c = self.chunks[i]
            if only_file and c["source"] != only_file:
                continue
            hits.append({**c, "score": float(s)})
            if len(hits) >= k:
                break
        return hits

    def info(self) -> str:
        if not self.files:
            return "No documents loaded."
        lines = [f"• {n}: {m['pages']} pages, {m['chunks']} chunks" for n, m in self.files.items()]
        lines.append(f"Total chunks: {len(self.chunks)}")
        return "\n".join(lines)


kb = KnowledgeBase()


# ---------------------------------------------------------------------
# LLM helpers
# ---------------------------------------------------------------------
def stream_llm(messages, model, temperature):
    resp = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, stream=True, max_tokens=1024
    )
    for chunk in resp:
        if chunk.choices and chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content


def format_context(hits):
    return "\n\n".join(f"[{h['source']} p.{h['page']}]\n{h['text']}" for h in hits)


def strip_sources(text: str) -> str:
    return text.split("\n\n---\n**Sources")[0]


# ---------------------------------------------------------------------
# UI handlers
# ---------------------------------------------------------------------
def handle_upload(files, chunk_size, overlap):
    if not files:
        return "❌ Please choose at least one PDF.", kb.info(), gr.update()
    logs = []
    for f in files:
        path = f if isinstance(f, str) else f.name
        try:
            logs.append(kb.add_pdf(path, int(chunk_size), int(overlap)))
        except Exception as e:
            logs.append(f"❌ {os.path.basename(path)}: {e}")
    choices = ["All documents"] + list(kb.files.keys())
    return "\n".join(logs), kb.info(), gr.update(choices=choices, value="All documents")


def clear_all():
    kb.reset()
    return "Cleared.", kb.info(), [], gr.update(choices=["All documents"], value="All documents")


def respond(message, history, k, temperature, model, scope, show_sources):
    history = list(history or [])
    if not message or not message.strip():
        yield history, ""
        return
    history.append({"role": "user", "content": message})

    if not kb.chunks:
        history.append({"role": "assistant", "content": "❌ Please upload and process a PDF first."})
        yield history, ""
        return

    # Short follow-ups ("and why?") retrieve better with the previous question attached
    query = message
    if len(message.split()) < 6:
        prev = [m["content"] for m in history[:-1] if m["role"] == "user"]
        if prev:
            query = prev[-1] + " " + message

    only = None if scope == "All documents" else scope
    hits = kb.search(query, int(k), only)
    if not hits:
        history.append({"role": "assistant", "content": "❌ No relevant passages found."})
        yield history, ""
        return

    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for m in history[:-1][-6:]:  # last few turns for conversational memory
        msgs.append({"role": m["role"], "content": strip_sources(m["content"])})
    msgs.append({
        "role": "user",
        "content": f"Context:\n{format_context(hits)}\n\nQuestion: {message}",
    })

    history.append({"role": "assistant", "content": ""})
    try:
        for delta in stream_llm(msgs, model, temperature):
            history[-1]["content"] += delta
            yield history, ""
    except Exception as e:
        history[-1]["content"] = f"❌ LLM error: {e}\n\nCheck your API key and model name."
        yield history, ""
        return

    if show_sources:
        src = "\n".join(
            f"- {h['source']} p.{h['page']} (match {h['score']:.2f}): {h['text'][:160].strip()}…"
            for h in hits
        )
        history[-1]["content"] += f"\n\n---\n**Sources**\n{src}"
    yield history, ""


def summarize(model, scope):
    if not kb.chunks:
        return "❌ Please upload and process a PDF first."
    pool = [c for c in kb.chunks if scope in ("All documents", None) or c["source"] == scope]
    # Sample evenly so long docs fit in the context window (~12k chars)
    budget, step = 12000, 1
    total = sum(len(c["text"]) for c in pool)
    if total > budget:
        step = int(np.ceil(total / budget))
    sample = pool[::step]
    text = "\n\n".join(f"[{c['source']} p.{c['page']}] {c['text']}" for c in sample)[:budget + 2000]
    msgs = [
        {"role": "system", "content": "You summarize documents accurately and concisely."},
        {"role": "user", "content": (
            "Summarize this document: a 2-3 sentence overview, then 5-8 key points as bullets, "
            "then any conclusions or action items.\n\n" + text)},
    ]
    try:
        return "".join(stream_llm(msgs, model, 0.2))
    except Exception as e:
        return f"❌ LLM error: {e}"


def suggest_questions(model, scope):
    if not kb.chunks:
        return "❌ Please upload and process a PDF first."
    pool = [c for c in kb.chunks if scope in ("All documents", None) or c["source"] == scope]
    sample = pool[:: max(1, len(pool) // 8)][:8]
    text = "\n\n".join(c["text"] for c in sample)[:6000]
    msgs = [{"role": "user", "content":
             "Based on this text, write 6 useful, specific questions a reader might ask. "
             "Return a numbered list only.\n\n" + text}]
    try:
        return "".join(stream_llm(msgs, model, 0.5))
    except Exception as e:
        return f"❌ LLM error: {e}"


def export_chat(history):
    if not history:
        return None
    lines = [f"# PDF Q&A chat - {datetime.now():%Y-%m-%d %H:%M}\n", f"Documents: {', '.join(kb.files) or 'none'}\n"]
    for m in history:
        who = "**You**" if m["role"] == "user" else "**Bot**"
        lines.append(f"{who}: {m['content']}\n")
    path = os.path.join(tempfile.gettempdir(), "pdf_qa_chat.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return path


# ---------------------------------------------------------------------
# Interface
# ---------------------------------------------------------------------
def build_ui():
    with gr.Blocks(title="PDF Q&A Bot") as demo:
        gr.Markdown("# 📄 PDF Q&A Bot\nUpload one or more PDFs, then chat with them. Answers cite file and page.")

        with gr.Row():
            with gr.Column(scale=1):
                files = gr.File(label="Upload PDFs", file_types=[".pdf"], file_count="multiple", type="filepath")
                with gr.Accordion("Chunking settings", open=False):
                    chunk_size = gr.Slider(300, 2000, value=1000, step=100, label="Chunk size (chars)")
                    overlap = gr.Slider(0, 500, value=200, step=50, label="Chunk overlap (chars)")
                process_btn = gr.Button("Process PDFs", variant="primary")
                status = gr.Textbox(label="Status", interactive=False, lines=3)
                doc_info = gr.Textbox(label="Loaded documents", interactive=False, lines=4)
                clear_btn = gr.Button("Clear everything")

                with gr.Accordion("Model settings", open=False):
                    model = gr.Dropdown(MODEL_CHOICES, value=DEFAULT_MODEL, allow_custom_value=True, label="Model")
                    temperature = gr.Slider(0, 1, value=0.2, step=0.05, label="Temperature")
                    k = gr.Slider(1, 10, value=4, step=1, label="Passages to retrieve (k)")
                    show_sources = gr.Checkbox(value=True, label="Show source passages")

            with gr.Column(scale=2):
                scope = gr.Dropdown(["All documents"], value="All documents", label="Search in")
                # Works on both Gradio 5 (needs type="messages") and Gradio 6 (option removed)
                try:
                    chatbot = gr.Chatbot(type="messages", height=480, label="Chat")
                except TypeError:
                    chatbot = gr.Chatbot(height=480, label="Chat")
                msg = gr.Textbox(placeholder="Ask something about your PDFs and press Enter…", label="Your question")
                with gr.Row():
                    send_btn = gr.Button("Ask", variant="primary")
                    clear_chat = gr.Button("Clear chat")
                    export_btn = gr.Button("Export chat (.md)")
                export_file = gr.File(label="Download", interactive=False)

                with gr.Tabs():
                    with gr.Tab("Summary"):
                        sum_btn = gr.Button("Summarize document")
                        sum_out = gr.Markdown()
                    with gr.Tab("Suggested questions"):
                        q_btn = gr.Button("Suggest questions")
                        q_out = gr.Markdown()

        process_btn.click(handle_upload, [files, chunk_size, overlap], [status, doc_info, scope])
        clear_btn.click(clear_all, None, [status, doc_info, chatbot, scope])

        chat_inputs = [msg, chatbot, k, temperature, model, scope, show_sources]
        msg.submit(respond, chat_inputs, [chatbot, msg])
        send_btn.click(respond, chat_inputs, [chatbot, msg])
        clear_chat.click(lambda: [], None, chatbot)
        export_btn.click(export_chat, chatbot, export_file)
        sum_btn.click(summarize, [model, scope], sum_out)
        q_btn.click(suggest_questions, [model, scope], q_out)
    return demo


if __name__ == "__main__" or True:  # `or True` so it also runs when pasted into a Colab cell
    demo = build_ui()
    demo.queue().launch(share=True, debug=False)

gemini API key: ··········
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ready.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1d8a3338583b335457.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
